# Feature Engineering and Forecast Preparation

## 1. Introduction

Feature engineering transforms raw data into meaningful variables that improve machine learning performance.

Since the dataset contains observations at 10-minute intervals, six records represent one hour.

This project uses historical electricity consumption, weather conditions, time-based features, lag features and rolling statistics to predict electricity demand one hour ahead.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("../data/Tetuan City power consumption.csv")

# Convert DateTime
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Sort chronologically
df = df.sort_values('DateTime').reset_index(drop=True)

# Define target zones
zones = [
    'Zone 1 Power Consumption',
    'Zone 2  Power Consumption',
    'Zone 3  Power Consumption'
]

# Display dataset information
print("Dataset loaded successfully!")
print("Original shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Original shape: (52416, 9)

Columns:
['DateTime', 'Temperature', 'Humidity', 'Wind Speed', 'general diffuse flows', 'diffuse flows', 'Zone 1 Power Consumption', 'Zone 2  Power Consumption', 'Zone 3  Power Consumption']


In [2]:
# Time-based features
df['Hour'] = df['DateTime'].dt.hour
df['Day'] = df['DateTime'].dt.day
df['Month'] = df['DateTime'].dt.month
df['DayOfWeek'] = df['DateTime'].dt.dayofweek
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)

# Cyclical time features
df['Hour_Sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_Cos'] = np.cos(2 * np.pi * df['Hour'] / 24)

df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)

# Forecast horizon: 1 hour = 6 records
forecast_steps = 6

# Create future target columns
for zone in zones:
    target_name = zone.replace(' Power Consumption', '_Target')
    df[target_name] = df[zone].shift(-forecast_steps)

# Create lag and rolling features
for zone in zones:
    prefix = zone.replace(' Power Consumption', '').replace(' ', '_')

    for lag in [0, 1, 6, 12, 144]:
        df[f'{prefix}_Lag_{lag}'] = df[zone].shift(lag)

    for window in [6, 12, 144]:
        df[f'{prefix}_Rolling_{window}'] = (
            df[zone].rolling(window=window).mean()
        )

# Remove rows containing NaN values from feature engineering
df = df.dropna().reset_index(drop=True)

print("Feature engineering completed!")
print("New dataset shape:", df.shape)

display(df.head())

Feature engineering completed!
New dataset shape: (52266, 45)


,DateTime,Temperature,Humidity,Wind Speed,general diffuse flows,diffuse flows,Zone 1 Power Consumption,Zone 2 Power Consumption,Zone 3 Power Consumption,Hour,...,Zone_2__Rolling_12,Zone_2__Rolling_144,Zone_3__Lag_0,Zone_3__Lag_1,Zone_3__Lag_6,Zone_3__Lag_12,Zone_3__Lag_144,Zone_3__Rolling_6,Zone_3__Rolling_12,Zone_3__Rolling_144
0,2017-01-02 00:00:00,11.01,75.8,0.076,0.033,0.163,26703.79747,18047.41641,17916.14458,0,...,21090.577507,17751.114488,17916.14458,18384.57831,20686.26506,23976.86747,20240.96386,19117.108433,20575.421686,17852.650602
1,2017-01-02 00:10:00,10.74,77.3,0.078,0.084,0.108,26169.11392,17507.59878,17349.39759,0,...,20560.182370,17738.145897,17349.39759,17916.14458,20397.10843,23415.90361,20131.08434,18609.156627,20069.879518,17833.333333
2,2017-01-02 00:20:00,10.47,78.2,0.078,0.088,0.130,25622.27848,17120.97264,16985.06024,0,...,20044.680850,17725.050658,16985.06024,17349.39759,19905.54217,22698.79518,19668.43373,18122.409638,19593.734939,17814.698795
3,2017-01-02 00:30:00,10.31,79.2,0.076,0.091,0.134,24972.15190,16785.41033,16586.02410,0,...,19528.875378,17714.108409,16586.02410,16985.06024,19252.04819,22322.89157,18899.27711,17678.072290,19115.662650,17798.634538
4,2017-01-02 00:40:00,10.63,79.3,0.079,0.037,0.152,24437.46835,16453.49544,16366.26506,0,...,19025.835865,17704.255319,16366.26506,16586.02410,18847.22892,21813.97590,18442.40964,17264.578313,18661.686747,17784.216867


In [3]:
# Save processed dataset
output_path = "../outputs/engineered_power_data.csv"

df.to_csv(output_path, index=False)

print("Engineered dataset saved successfully!")
print("Saved location:", output_path)
print("Final records:", len(df))
print("Total features and columns:", len(df.columns))

Engineered dataset saved successfully!
Saved location: ../outputs/engineered_power_data.csv
Final records: 52266
Total features and columns: 45
